# Progetto Machine Learning: Previsione del Prezzo Unico Nazionale (PUN)
## 4. Modeling & 5. Evaluation (Multi-Horizon Forecasting)

**Corso:** Machine Learning e Data Mining (Modulo 2) — Prof. Matteo Francia  
**Dataset:** Prezzi orari MGP (GME) & Dati Meteo Zonali (Open-Meteo) 2024–2025  

---

### Obiettivi e Metodologia CRISP-DM

Questo notebook copre la seconda parte del progetto, focalizzata sulle fasi **4. Modeling** e **5. Evaluation** della metodologia CRISP-DM:

1. **Integrazione dei Dataset Processati:** Importazione delle matrici pulite (`X_train`, `y_train`, `X_test`, `y_test`) generate nella fase di Data Preparation (`01_data_ingestion.ipynb`).
2. **Definizione delle Baseline:** Addestramento di modelli di riferimento (Naive e Regressione Ridge/Lasso) per definire la soglia minima di performance accettabile.
3. **Modellazione Predittiva Avanzata:** Implementazione e benchmarking di algoritmi di Machine Learning (Decision Tree, Random Forest, XGBoost, MLP) per previsioni multi-orizzonte (24h, 48h, 72h).
4. **Valutazione delle Performance:** Analisi e confronto delle metriche di errore ($\text{MAE}$, $\text{RMSE}$, $\text{MAPE}$) sul test set per identificare il modello con il miglior grado di generalizzazione.

In [137]:
# ==============================================================================
# SETUP AMBIENTE E IMPORT LIBRERIE
# ==============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

# Gestione Working Directory (Google Colab vs Locale Mac)
if os.path.exists("/content"):
    if not os.path.exists("progetto-ml-pun"):
        # !git clone https://github.com/erloyz/progetto-ml-pun.git
        # %cd /content/progetto-ml-pun
        pass
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("📍 Working Directory corrente:", os.getcwd())

# --- MANIPOLAZIONE DATI E GRAFICI ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- MACHINE LEARNING (Scikit-Learn & XGBoost) ---
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb

# MODIFICA MULTI-OUTPUT: Import fondamentale per adattare modelli Single-Output (es. Ridge, Lasso) a 3 target
from sklearn.multioutput import MultiOutputRegressor

# --- METRICHE E VALIDAZIONE TEMPORALE ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import randint, uniform

print("✅ Ambiente configurato e librerie caricate con successo!")

📍 Working Directory corrente: /Users/erloyz/Desktop/uni/dm-ml/ml/progetto-ml-pun
✅ Ambiente configurato e librerie caricate con successo!


## 1. Caricamento Dataset Processati
Caricamento dei dataset di Training e Test (`X_train`, `X_test`, `y_train`, `y_test`) esportati dalla pipeline di ingestion[cite: 9, 10]. Il data split è stato effettuato in ordine cronologico per evitare Data Leakage[cite: 9].

In [138]:
# ==============================================================================
# 1. CARICAMENTO DATASET PROCESSATI
# ==============================================================================

X_train = pd.read_csv("data/processed/X_train.csv")
y_train = pd.read_csv("data/processed/y_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")
y_test = pd.read_csv("data/processed/y_test.csv")

# MODIFICA MULTI-OUTPUT: Manteniamo y come matrice 2D (N_samples, 3) 
# Rimuoviamo .ravel() perché le 3 colonne (target_24h, target_48h, target_72h) servono tutte ai modelli!
y_train_vec = y_train.values
y_test_vec = y_test.values

print("📌 Dati caricati correttamente!")
print(f"- X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"- X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
# MODIFICA MULTI-OUTPUT: Controllo valori nulli coerente per la matrice 2D
print(f"- X_train nulli: {X_train.isnull().sum().sum()} | y_train nulli: {np.isnan(y_train_vec).sum()}")
print(f"- X_test nulli:  {X_test.isnull().sum().sum()} | y_test nulli:  {np.isnan(y_test_vec).sum()}")

📌 Dati caricati correttamente!
- X_train shape: (12959, 21) | y_train shape: (12959, 3)
- X_test shape:  (4345, 21)  | y_test shape:  (4345, 3)
- X_train nulli: 0 | y_train nulli: 0
- X_test nulli:  0 | y_test nulli:  0


In [139]:
# ==============================================================================
# VERIFICA STRUTTURA DELLE FEATURE E DEI TARGET
# ==============================================================================
print(f"Numero totale di Feature di input (N): {X_train.shape[1]}\n")
print("Lista delle variabili di input:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")

# MODIFICA MULTI-OUTPUT: Stampa di verifica per i 3 target
print(f"\nNumero totale di Target Multi-Orizzonte: {y_train.shape[1]}")
print("Colonne Target:", list(y_train.columns))

Numero totale di Feature di input (N): 21

Lista delle variabili di input:
   1. Data
   2. Ora
   3. temperature_2m_CentroSud
   4. temperature_2m_Nord
   5. wind_speed_10m_CentroSud
   6. wind_speed_10m_Nord
   7. shortwave_radiation_CentroSud
   8. shortwave_radiation_Nord
   9. ora_num
  10. mese_num
  11. giorno_settimana
  12. sin_ora
  13. cos_ora
  14. sin_mese
  15. cos_mese
  16. is_weekend
  17. pun_lag_24
  18. pun_lag_48
  19. pun_lag_72
  20. pun_lag_168
  21. pun_rolling_mean_24h

Numero totale di Target Multi-Orizzonte: 3
Colonne Target: ['target_24h', 'target_48h', 'target_72h']


## 2. Feature Scaling (Prevenzione Data Leakage)
Applicazione del **MinMaxScaler [0, 1]** per le feature numeriche destinate a modelli sensibili alla scala (Regressione Lineare, Ridge, Lasso, MLPRegressor)[cite: 9]. 
> **Nota Anti-Leakage:** Il metodo `.fit()` viene calcolato **esclusivamente su `X_train`**; su `X_test` viene applicato soltanto il `.transform()` usando i minimi e massimi appresi dallo storico[cite: 9].

In [140]:
# ==============================================================================
# FEATURE SCALING (MinMaxScaler per modelli sensibili alla scala)
# ==============================================================================
# Applico MinMaxScaler [0, 1] per normalizzare le feature per modelli come Ridge e MLPRegressor.
# Il .fit() viene eseguito ESCLUSIVAMENTE su X_train per imparare min e max storici,
# mentre su X_test si applica soltanto il .transform() per prevenire il Data Leakage.

from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. Escludiamo le colonne di tipo data/stringa prima dello scaling
cols_to_drop = [col for col in ['Data', 'timestamp'] if col in X_train.columns]

X_train_num = X_train.drop(columns=cols_to_drop)
X_test_num = X_test.drop(columns=cols_to_drop)

# 2. Inizializzazione dello Scaler per le feature numeriche
scaler_X = MinMaxScaler(feature_range=(0, 1))

# 3. Fit su X_train numerico | Transform sia su Train che su Test
X_train_scaled = scaler_X.fit_transform(X_train_num)
X_test_scaled = scaler_X.transform(X_test_num)

# 4. Conversione in DataFrame mantenendo i nomi delle colonne
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_num.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_num.columns)

print("✅ Feature Scaling (MinMaxScaler) completato correttamente!")
print(f"Dimensioni X_train scalato: {X_train_scaled.shape}")
print(f"Dimensioni X_test scalato:  {X_test_scaled.shape}")

✅ Feature Scaling (MinMaxScaler) completato correttamente!
Dimensioni X_train scalato: (12959, 20)
Dimensioni X_test scalato:  (4345, 20)


## 3. Definizione del Framework di Valutazione
Definizione della funzione `calcola_metriche()` per la registrazione unificata delle prestazioni del modello sui prezzi reali in €/MWh tramite **MAE**, **RMSE** e **MAPE**[cite: 9].

In [141]:
# ==============================================================================
# 3. DEFINIZIONE DEL FRAMEWORK DI VALUTAZIONE MULTI-ORIZZONTE
# ==============================================================================

def calcola_metriche(y_true, y_pred, nome_modello="Modello"):
    """
    Calcola e restituisce MAE, RMSE e MAPE per ciascun orizzonte temporale (24h, 48h, 72h)
    e la media globale multi-output.
    """
    orizzonti = ['24h', '48h', '72h']
    risultati_orizzonti = {}
    
    print(f"\n==================================================")
    print(f"   PERFORMANCE MODELLO: {nome_modello}")
    print(f"==================================================")
    
    # MODIFICA MULTI-OUTPUT: Calcolo metriche per singolo orizzonte
    for i, h in enumerate(orizzonti):
        mae_h = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse_h = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        mape_h = mean_absolute_percentage_error(y_true[:, i], y_pred[:, i]) * 100
        
        risultati_orizzonti[h] = {'MAE': mae_h, 'RMSE': rmse_h, 'MAPE': mape_h}
        print(f"🔹 Orizzonte +{h}: MAE = {mae_h:.2f} €/MWh | RMSE = {rmse_h:.2f} €/MWh | MAPE = {mape_h:.2f}%")
    
    # MODIFICA MULTI-OUTPUT: Calcolo media complessiva sui 3 orizzonti
    mae_mean = mean_absolute_error(y_true, y_pred)
    rmse_mean = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_mean = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    print(f"--------------------------------------------------")
    print(f"   MEDIA COMPLESSIVA (Multi-Step 72h):")
    print(f"   MAE:  {mae_mean:.4f} €/MWh")
    print(f"   RMSE: {rmse_mean:.4f} €/MWh")
    print(f"   MAPE: {mape_mean:.2f} %")
    print(f"==================================================\n")
    
    return {
        "Modello": nome_modello,
        "MAE": mae_mean,
        "RMSE": rmse_mean,
        "MAPE": mape_mean,
        "Dettaglio_Orizzonti": risultati_orizzonti
    }

# Lista per collezionare i risultati dei vari modelli e confrontarli alla fine
risultati_benchmark = []

## 4. Modeling & Benchmarking

### 4.0 Baseline Model (Persistence Lag-24)
Modello di riferimento naïve: ipotizza che il prezzo all'ora $t$ sia uguale a quello del giorno precedente alla stessa ora ($t-24$)[cite: 9]. Serve come soglia minima di accuratezza per validare i modelli ML[cite: 9].

In [142]:
# ==============================================================================
# 4.0 BASELINE MODEL (Persistence Settimanale - Lag 168h)
# ==============================================================================

# Sfruttiamo la stagionalità settimanale (168 ore) come baseline robusta 
# per tutti e tre gli orizzonti (24h, 48h, 72h)
y_pred_baseline = np.column_stack([
    X_test['pun_lag_168'],  # Colonna 0: Previsione per t+24h
    X_test['pun_lag_168'],  # Colonna 1: Previsione per t+48h
    X_test['pun_lag_168']   # Colonna 2: Previsione per t+72h
])

# Calcolo metriche per la Baseline multi-orizzonte
res_baseline = calcola_metriche(y_test_vec, y_pred_baseline, nome_modello="Baseline (Persistence Settimanale 168h)")
risultati_benchmark.append(res_baseline)


   PERFORMANCE MODELLO: Baseline (Persistence Settimanale 168h)
🔹 Orizzonte +24h: MAE = 13.45 €/MWh | RMSE = 19.85 €/MWh | MAPE = 14.52%
🔹 Orizzonte +48h: MAE = 15.05 €/MWh | RMSE = 22.06 €/MWh | MAPE = 17.06%
🔹 Orizzonte +72h: MAE = 15.43 €/MWh | RMSE = 22.65 €/MWh | MAPE = 17.45%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  14.6455 €/MWh
   RMSE: 21.5554 €/MWh
   MAPE: 16.35 %



### 4.1 Modelli Statistici e Lineari (Ridge & Lasso Regression)
Addestramento dei modelli di regressione lineare regolarizzata con penalità $L_2$ (Ridge) ed $L_1$ (Lasso) utilizzando le matrici di feature scalate (`X_train_scaled`, `X_test_scaled`) per prevenire squilibri tra le variabili esogene e le feature cicliche.

* **Ridge ($L_2$):** Applica una penalità quadratica sui pesi, riducendoli in modo uniforme senza annullarli del tutto.
* **Lasso ($L_1$):** Applica una penalità sul valore assoluto dei coefficienti, azzerando le variabili meno rilevanti e fungendo da metodo integrato di **Feature Selection automatica**.

In [143]:
# ==============================================================================
# MODELLO 1: RIDGE REGRESSION (REGOLARIZZAZIONE L2)
# ==============================================================================
from sklearn.linear_model import Ridge

# 1. Inizializzazione e fit del modello sui dati SCALATI
model_ridge = Ridge(alpha=1.0, random_state=42)
model_ridge.fit(X_train_scaled, y_train_vec)

# 2. Previsione sul Test Set scalato
y_pred_ridge = model_ridge.predict(X_test_scaled)

# 3. Calcolo e salvataggio delle metriche
res_ridge = calcola_metriche(y_test_vec, y_pred_ridge, nome_modello="Ridge Regression")
risultati_benchmark.append(res_ridge)


   PERFORMANCE MODELLO: Ridge Regression
🔹 Orizzonte +24h: MAE = 11.23 €/MWh | RMSE = 15.85 €/MWh | MAPE = 13.42%
🔹 Orizzonte +48h: MAE = 11.68 €/MWh | RMSE = 15.77 €/MWh | MAPE = 13.63%
🔹 Orizzonte +72h: MAE = 11.88 €/MWh | RMSE = 16.56 €/MWh | MAPE = 14.14%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  11.6001 €/MWh
   RMSE: 16.0612 €/MWh
   MAPE: 13.73 %



In [144]:
# ==============================================================================
# MODELLO 1B: LASSO REGRESSION (REGOLARIZZAZIONE L1)
# ==============================================================================
from sklearn.linear_model import Lasso

model_lasso = Lasso(alpha=0.01, random_state=42)
model_lasso.fit(X_train_scaled, y_train_vec)

y_pred_lasso = model_lasso.predict(X_test_scaled)

res_lasso = calcola_metriche(y_test_vec, y_pred_lasso, nome_modello="Lasso Regression")
risultati_benchmark.append(res_lasso)


   PERFORMANCE MODELLO: Lasso Regression
🔹 Orizzonte +24h: MAE = 11.21 €/MWh | RMSE = 15.82 €/MWh | MAPE = 13.38%
🔹 Orizzonte +48h: MAE = 11.67 €/MWh | RMSE = 15.75 €/MWh | MAPE = 13.62%
🔹 Orizzonte +72h: MAE = 11.87 €/MWh | RMSE = 16.55 €/MWh | MAPE = 14.13%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  11.5858 €/MWh
   RMSE: 16.0461 €/MWh
   MAPE: 13.71 %



### 4.2 Modelli Tree-Based ed Ensemble Learning (Random Forest & XGBoost)
Addestramento di modelli non parametrici basati su alberi decisionali (Random Forest e XGBoost)[cite: 9, 13]. Gli alberi gestiscono nativamente relazioni non lineari, interazioni complesse tra le feature e sono invarianti rispetto allo scaling numerico delle variabili[cite: 9, 13].

In [145]:
# ==============================================================================
# MODELLO 2: RANDOM FOREST REGRESSOR
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor

# 1. Utilizziamo X_train_num e X_test_num (senza la colonna della data)
model_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
model_rf.fit(X_train_num, y_train_vec)

# 2. Previsione sul Test Set
y_pred_rf = model_rf.predict(X_test_num)

# 3. Calcolo e salvataggio delle metriche
res_rf = calcola_metriche(y_test_vec, y_pred_rf, nome_modello="Random Forest")
risultati_benchmark.append(res_rf)


   PERFORMANCE MODELLO: Random Forest
🔹 Orizzonte +24h: MAE = 9.33 €/MWh | RMSE = 13.39 €/MWh | MAPE = 10.93%
🔹 Orizzonte +48h: MAE = 10.06 €/MWh | RMSE = 14.50 €/MWh | MAPE = 11.89%
🔹 Orizzonte +72h: MAE = 10.34 €/MWh | RMSE = 14.96 €/MWh | MAPE = 12.44%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  9.9073 €/MWh
   RMSE: 14.2994 €/MWh
   MAPE: 11.75 %



In [146]:
# ==============================================================================
# XGBOOST HYPERPARAMETER TUNING (TIMESERIES CROSS-VALIDATION) - STILE PROF. FRANCIA
# ==============================================================================
import numpy as np
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# 1. Definizione della griglia probabilistica (usando scipy.stats come nelle slide)
param_dist = {
    'n_estimators': randint(100, 600),       # Estrae numeri interi uniformi tra 100 e 599
    'max_depth': randint(3, 10),              # Estrae numeri interi uniformi tra 3 e 9
    'learning_rate': uniform(0.01, 0.09),     # Estrae un float continuo tra 0.01 e 0.10
    'subsample': uniform(0.6, 0.4),           # Estrae un float continuo tra 0.60 e 1.00
    'colsample_bytree': uniform(0.6, 0.4),    # Estrae un float continuo tra 0.60 e 1.00
    'reg_alpha': uniform(0.0, 1.0),           # Regolarizzazione L1 continua [0.0, 1.0]
    'reg_lambda': uniform(1.0, 4.0)           # Regolarizzazione L2 continua [1.0, 5.0]
}

# 2. TimeSeriesSplit per rispettare la sequenza temporale ed evitare Data Leakage
tscv = TimeSeriesSplit(n_splits=3)

# 3. RandomizedSearchCV istanziata con le distribuzioni di scipy.stats
xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,                                # 15 combinazioni casuali per limitare i tempi
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    random_state=42,
    n_jobs=-1
)

# 4. Fit sulle matrici numeriche di addestramento (X_train_num, y_train_vec)
xgb_search.fit(X_train_num, y_train_vec)

print("📌 Migliori Iperparametri trovati per XGBoost:")
print(xgb_search.best_params_)

# 5. Valutazione del miglior modello sul Test Set out-of-sample
model_xgb_tuned = xgb_search.best_estimator_
y_pred_xgb_tuned = model_xgb_tuned.predict(X_test_num)

res_xgb_tuned = calcola_metriche(y_test_vec, y_pred_xgb_tuned, nome_modello="XGBoost (Tuned)")
print(res_xgb_tuned)

📌 Migliori Iperparametri trovati per XGBoost:
{'colsample_bytree': np.float64(0.8270801311279966), 'learning_rate': np.float64(0.012818196321000272), 'max_depth': 4, 'n_estimators': 317, 'reg_alpha': np.float64(0.44975413336976566), 'reg_lambda': np.float64(2.5806009440072577), 'subsample': np.float64(0.9706635463175177)}

   PERFORMANCE MODELLO: XGBoost (Tuned)
🔹 Orizzonte +24h: MAE = 9.17 €/MWh | RMSE = 13.29 €/MWh | MAPE = 10.97%
🔹 Orizzonte +48h: MAE = 9.78 €/MWh | RMSE = 14.18 €/MWh | MAPE = 11.70%
🔹 Orizzonte +72h: MAE = 10.16 €/MWh | RMSE = 14.64 €/MWh | MAPE = 12.15%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  9.7019 €/MWh
   RMSE: 14.0513 €/MWh
   MAPE: 11.60 %

{'Modello': 'XGBoost (Tuned)', 'MAE': 9.701895795184464, 'RMSE': np.float64(14.051270798508863), 'MAPE': 11.604983174436123, 'Dettaglio_Orizzonti': {'24h': {'MAE': 9.172891883570207, 'RMSE': np.float64(13.291563641322705), 'MAPE': 10.965237920433822}, '48h': {'MAE'

### 4.3 Deep Learning / Reti Neurali (MLPRegressor)
Addestramento di una Rete Neurale Multi-Layer Perceptron (MLP) sulle matrici di feature scalate (`X_train_scaled`, `X_test_scaled`). Le reti neurali richiedono dati normalizzati in scala [0, 1] per garantire la stabilità numerica e la rapida convergenza della discesa del gradiente (algoritmo Adam)[cite: 10].

In [147]:
# ==============================================================================
# MODELLO 4: MULTI-LAYER PERCEPTRON (MLP REGRESSOR)
# ==============================================================================
from sklearn.neural_network import MLPRegressor

# 1. Inizializzazione della Rete Neurale con 2 hidden layer (64 e 32 neuroni)
model_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,  # Previene l'overfitting interrompendo l'addestramento se l'errore non migliora
    n_iter_no_change=10
)

# 2. Fit sui dati SCALATI
model_mlp.fit(X_train_scaled, y_train_vec)

# 3. Previsione sul Test Set scalato
y_pred_mlp = model_mlp.predict(X_test_scaled)

# 4. Calcolo e salvataggio delle metriche
res_mlp = calcola_metriche(y_test_vec, y_pred_mlp, nome_modello="MLP Regressor")
risultati_benchmark.append(res_mlp)


   PERFORMANCE MODELLO: MLP Regressor
🔹 Orizzonte +24h: MAE = 11.11 €/MWh | RMSE = 15.34 €/MWh | MAPE = 13.09%
🔹 Orizzonte +48h: MAE = 11.21 €/MWh | RMSE = 15.27 €/MWh | MAPE = 12.92%
🔹 Orizzonte +72h: MAE = 11.56 €/MWh | RMSE = 15.59 €/MWh | MAPE = 13.31%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  11.2955 €/MWh
   RMSE: 15.3994 €/MWh
   MAPE: 13.11 %



### 4.4 Support Vector Regression (SVR)
Addestramento del modello **Support Vector Regressor** con kernel RBF (Radial Basis Function) sulle matrici di feature scalate (`X_train_scaled`, `X_test_scaled`).

La funzione di perdita $\epsilon$-insensitive dell'SVR rende l'algoritmo particolarmente robusto ai picchi isolati di prezzo e al rumore di fondo dei mercati energetici. Come evidenziato dalla letteratura di settore, le Support Vector Machines si posizionano costantemente tra le architetture top-performer per la previsione del PUN nel Day-Ahead Market.

In [148]:
# ==============================================================================
# MODELLO 5: SUPPORT VECTOR REGRESSION (SVR con MultiOutputRegressor)
# ==============================================================================
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

# 1. Inizializzazione SVR con Kernel RBF avvolto per gestire il Multi-Output
model_svr = MultiOutputRegressor(SVR(kernel='rbf', C=12.0, epsilon=0.1))

# 2. Fit sui dati scalati
model_svr.fit(X_train_scaled, y_train_vec)

# 3. Previsione sul Test Set scalato
y_pred_svr = model_svr.predict(X_test_scaled)

# 4. Calcolo e salvataggio delle metriche
res_svr = calcola_metriche(y_test_vec, y_pred_svr, nome_modello="SVR (RBF Kernel)")
risultati_benchmark.append(res_svr)


   PERFORMANCE MODELLO: SVR (RBF Kernel)
🔹 Orizzonte +24h: MAE = 10.41 €/MWh | RMSE = 14.61 €/MWh | MAPE = 12.48%
🔹 Orizzonte +48h: MAE = 10.94 €/MWh | RMSE = 15.22 €/MWh | MAPE = 13.03%
🔹 Orizzonte +72h: MAE = 11.84 €/MWh | RMSE = 16.55 €/MWh | MAPE = 14.07%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  11.0654 €/MWh
   RMSE: 15.4806 €/MWh
   MAPE: 13.19 %



In [149]:
# ==============================================================================
# MODELLO 6: LINEAR SUPPORT VECTOR REGRESSION (SVR Lineare con MultiOutputRegressor)
# ==============================================================================
from sklearn.svm import LinearSVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Selezioniamo SOLO le colonne numeriche da X_train e X_test (escludendo date o stringhe)
X_train_num = X_train.select_dtypes(include=['number'])
X_test_num = X_test.select_dtypes(include=['number'])

# 2. Scaling con StandardScaler sulle sole feature numeriche
scaler_std = StandardScaler()
X_train_scaled = scaler_std.fit_transform(X_train_num)
X_test_scaled = scaler_std.transform(X_test_num)

# 3. Inizializzazione LinearSVR avvolto nel MultiOutputRegressor
model_svr_linear = MultiOutputRegressor(
    LinearSVR(
        C=10.0, 
        epsilon=0.01, 
        random_state=42, 
        max_iter=45000
    )
)

# 4. Fit sui dati STANDARDIZZATI
model_svr_linear.fit(X_train_scaled, y_train_vec)

# 5. Previsione e metriche sul Test Set
y_pred_svr_linear = model_svr_linear.predict(X_test_scaled)

res_svr_linear = calcola_metriche(y_test_vec, y_pred_svr_linear, nome_modello="SVR (Linear Kernel)")
risultati_benchmark.append(res_svr_linear)


   PERFORMANCE MODELLO: SVR (Linear Kernel)
🔹 Orizzonte +24h: MAE = 10.77 €/MWh | RMSE = 15.77 €/MWh | MAPE = 13.12%
🔹 Orizzonte +48h: MAE = 11.13 €/MWh | RMSE = 15.78 €/MWh | MAPE = 13.41%
🔹 Orizzonte +72h: MAE = 11.56 €/MWh | RMSE = 16.80 €/MWh | MAPE = 14.06%
--------------------------------------------------
   MEDIA COMPLESSIVA (Multi-Step 72h):
   MAE:  11.1540 €/MWh
   RMSE: 16.1246 €/MWh
   MAPE: 13.53 %



---

## 5. SEZIONE 5: VALUTAZIONE E CONFRONTO BENCHMARK (CRISP-DM EVALUATION)

In questa sezione finale del workflow CRISP-DM, viene effettuata la valutazione comparativa out-of-sample di tutte le architetture sviluppate[cite: 62, 63]. I modelli vengono posti a confronto rispetto alla **Baseline di persistenza (Lag-24)**[cite: 62].

### 📐 Metriche di Valutazione del Dominio
Per valutare la bontà predittiva e la stabilità delle stime sul Prezzo Unico Nazionale (PUN), utilizziamo quattro metriche complementari[cite: 62]:

1. **MAE (Mean Absolute Error) [€/MWh]:** Misura la deviazione media assoluta delle stime espresso nella valuta del mercato[cite: 62].
2. **RMSE (Root Mean Squared Error) [€/MWh]:** Penalizza in modo quadratico i grandi errori di previsione, fungendo da metrica guida principale[cite: 62].
3. **MAPE (Mean Absolute Percentage Error) [%]:** Fornisce l'errore relativo espresso in percentuale rispetto al prezzo reale[cite: 62].
4. **Anomaly Index (>20%) [%]:** Metrica specifica del settore energetico ispirata alla letteratura scientifica (Hosseini Imani et al., IEEE 2021). Calcola la percentuale di ore nel Test set in cui l'errore di stima assoluto supera la soglia di tolleranza del 20% del PUN reale, misurando l'incidenza delle "code di errore" e dei picchi non catturati.

### 🏆 Modelli nel Benchmark
Il benchmark include 8 configurazioni modellate nell'ambito della sperimentazione:
* **Baseline:** Persistence Model ($t-24$)[cite: 62].
* **Modelli Lineari & Regolarizzati:** Ridge Regression ($L_2$), Lasso Regression ($L_1$), Support Vector Regression con Linear Kernel (`LinearSVR`) e SVR RBF Kernel[cite: 62].
* **Modelli Tree-Based ed Ensemble:** Random Forest Regressor, XGBoost Baseline ed **XGBoost Tuned** (ottimizzato via `RandomizedSearchCV` e `TimeSeriesSplit`)[cite: 62].
* **Deep Learning:** Multi-Layer Perceptron (`MLPRegressor`)[cite: 62].

---

In [151]:
# ==============================================================================
# TABELLA BENCHMARK FINALE MULTI-ORIZZONTE DETTAGLIATA
# ==============================================================================
import pandas as pd
import numpy as np

# Dizionario contenente tutte le previsioni generate dai vari modelli
# (Assicurati che ogni y_pred abbia forma (N_campioni, 3))
dizionario_previsioni = {
    'Baseline (Persistenza 24h)': y_pred_baseline,
    'Ridge Regression': y_pred_ridge,
    'Lasso Regression': y_pred_lasso,
    'Random Forest': y_pred_rf,
    'XGBoost (Tuned)': y_pred_xgb_tuned,
    'MLP Regressor': y_pred_mlp,
    'SVR (RBF Kernel)': y_pred_svr,
    'SVR (Linear Kernel)': y_pred_svr_linear
}

orizzonti = ['+24h', '+48h', '+72h']
righe_benchmark = []

for nome_modello, y_pred in dizionario_previsioni.items():
    # Metriche globali (su tutti e 3 gli orizzonti insieme)
    mae_globale = np.mean(np.abs(y_test_vec - y_pred))
    rmse_globale = np.sqrt(np.mean((y_test_vec - y_pred) ** 2))
    mape_globale = np.mean(np.abs((y_test_vec - y_pred) / (y_test_vec + 1e-8))) * 100
    
    # Dizionario base per il modello
    riga = {
        'Modello': nome_modello,
        'MAE Globale': round(mae_globale, 2),
        'RMSE Globale': round(rmse_globale, 2),
        'MAPE (%) Globale': round(mape_globale, 2)
    }
    
    # Calcolo metriche specifiche per ogni singolo orizzonte (colonna 0=+24, 1=+48, 2=+72)
    for i, h in enumerate(orizzonti):
        y_true_h = y_test_vec[:, i] if hasattr(y_test_vec, 'ndim') and y_test_vec.ndim > 1 else y_test_vec
        # Gestione sicura nel caso y_pred sia array 2D
        y_pred_h = y_pred[:, i] if hasattr(y_pred, 'ndim') and y_pred.ndim > 1 else y_pred
        
        mae_h = np.mean(np.abs(y_true_h - y_pred_h))
        rmse_h = np.sqrt(np.mean((y_true_h - y_pred_h) ** 2))
        
        riga[f'MAE {h}'] = round(mae_h, 2)
        riga[f'RMSE {h}'] = round(rmse_h, 2)

    righe_benchmark.append(riga)

# Creazione del DataFrame finale ordinato per RMSE Globale crescente
df_benchmark_dettagliato = pd.DataFrame(righe_benchmark).sort_values(by='RMSE Globale').reset_index(drop=True)

print("🏆 CLASSIFICA FINALE MULTI-ORIZZONTE (Day-Ahead Benchmark):")
display(df_benchmark_dettagliato)

🏆 CLASSIFICA FINALE MULTI-ORIZZONTE (Day-Ahead Benchmark):


,Modello,MAE Globale,RMSE Globale,MAPE (%) Globale,MAE +24h,RMSE +24h,MAE +48h,RMSE +48h,MAE +72h,RMSE +72h
0,XGBoost (Tuned),9.70,14.05,11.60,9.17,13.29,9.78,14.18,10.16,14.64
1,Random Forest,9.91,14.30,11.75,9.33,13.39,10.06,14.50,10.34,14.96
2,MLP Regressor,11.30,15.40,13.11,11.11,15.34,11.21,15.27,11.56,15.59
3,SVR (RBF Kernel),11.07,15.48,13.19,10.41,14.61,10.94,15.22,11.84,16.55
4,Lasso Regression,11.59,16.05,13.71,11.21,15.82,11.67,15.75,11.87,16.55
5,Ridge Regression,11.60,16.06,13.73,11.23,15.85,11.68,15.77,11.88,16.56
6,SVR (Linear Kernel),11.15,16.12,13.53,10.77,15.77,11.13,15.78,11.56,16.80
7,Baseline (Persistenza 24h),14.65,21.56,16.35,13.45,19.85,15.05,22.06,15.43,22.65
